In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')


In [4]:
# Resolve raw-data directory from either project root or notebook directory
candidate_dirs = [Path('data/raw'), Path('../../data/raw')]
data_dir = next((p for p in candidate_dirs if p.exists()), None)
if data_dir is None:
    raise FileNotFoundError('Cannot find data/raw directory from current working directory.')

orders = pd.read_csv(data_dir / 'orders.csv')
order_products_prior = pd.read_csv(data_dir / 'order_products__prior.csv')
order_products_train = pd.read_csv(data_dir / 'order_products__train.csv')
products = pd.read_csv(data_dir / 'products.csv')
aisles = pd.read_csv(data_dir / 'aisles.csv')
departments = pd.read_csv(data_dir / 'departments.csv')

tables = {
    'orders': orders,
    'order_products_prior': order_products_prior,
    'order_products_train': order_products_train,
    'products': products,
    'aisles': aisles,
    'departments': departments,
}

print(f'Loaded {len(tables)} tables from: {data_dir.resolve()}')

Loaded 6 tables from: /home/ppt/Desktop/DataAnalyst/Instacart-Market-Basket-Analysis/data/raw


In [5]:
# Check columns first
print('order_products_prior columns:', order_products_prior.columns.tolist())
print('orders columns:', orders.columns.tolist())
print()

# Merge orders with order_products_prior to get user_id
df = orders[['order_id', 'user_id']].merge(order_products_prior, on='order_id')

# Create feature: Count how many times each user reordered each product
customer_product_reorder_count = df.groupby(['user_id', 'product_id'])['reordered'].sum().reset_index()
customer_product_reorder_count.rename(columns={'reordered': 'reorder_count'}, inplace=True)

print(customer_product_reorder_count.head())
print(f'Shape: {customer_product_reorder_count.shape}')

order_products_prior columns: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']
orders columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']

   user_id  product_id  reorder_count
0        1         196              9
1        1       10258              8
2        1       10326              0
3        1       12427              9
4        1       13032              2
Shape: (13307953, 3)


In [6]:
# Save the feature to CSV
output_dir = Path('data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / 'customer_product_reorder_count.csv'
customer_product_reorder_count.to_csv(output_file, index=False)
print(f'✓ Saved to: {output_file.resolve()}')

✓ Saved to: /home/ppt/Desktop/DataAnalyst/Instacart-Market-Basket-Analysis/notebooks/01_Data_Understanding/data/processed/customer_product_reorder_count.csv
